# Direct Probing Experiment

**Direct probing** using `JudgeConfig` with forced classes. We provide each persona's
conversation history + a probing question as `messages` in a `JudgeSubject`, and the
judge must classify the response into one of the allowed options.

- Gender: forced classes `["Male", "Female"]`
- Region: forced classes = the 5 selected regions

**Sampling:** 50 personas — 25 male, 25 female, 5 per region (5 regions).

> **Note:** interactive prototype. The canonical paper runs use the
> `run_direct_probing*.py` scripts in this directory (see `README.md`).


## 1. Setup

In [ ]:
import json
import random
from collections import defaultdict
from pathlib import Path

import pandas as pd

from inference import (
    JudgeConfig,
    JudgeExecutionConfig,
    JudgeSubject,
    create_client,
    run_judges,
)
from inference.experiments import ExperimentConfig, ExperimentRunner, to_analysis_dataframe
from inference.judges.log import JudgeLogger


def _repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "config" / "inference.example.yaml").exists():
            return p
    return Path.cwd()


REPO_ROOT = _repo_root()
CONFIG_PATH = REPO_ROOT / "config" / "inference.yaml"
PERSONAS_PATH = REPO_ROOT / "src" / "generate_backgrounds" / "data" / "personas" / "personas.jsonl"

client = create_client(CONFIG_PATH)

## 2. Load and sample personas

In [ ]:
all_personas = []
with open(PERSONAS_PATH) as f:
    for line in f:
        all_personas.append(json.loads(line))

print(f"Total personas loaded: {len(all_personas)}")

grouped: dict[tuple[str, str], list[dict]] = defaultdict(list)
for p in all_personas:
    gender = p["persona"].get("Gender")
    region = p["persona"].get("Region")
    if gender and region:
        grouped[(gender, region)].append(p)

all_regions = sorted({region for (_, region) in grouped})
print(f"Available regions ({len(all_regions)}): {all_regions}")

In [ ]:
SELECTED_REGIONS = all_regions
SAMPLE_PER_GROUP = 10000
random.seed(123)

# Change RUN_TAG to start a new independent run.
# Keep it the same to resume (ExperimentRunner handles resume automatically).
RUN_TAG = "gemma4full"
EXPERIMENT_MODEL = "gemma-4-31b_paid"  # model that acts as participant in stage 1

EXPERIMENT_NAME = f"direct-probing-combined-{RUN_TAG}" if RUN_TAG else "direct-probing-combined"

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Model     : {EXPERIMENT_MODEL}")

sampled_personas = []
for region in SELECTED_REGIONS:
    for gender in ["Male", "Female"]:
        pool = list(grouped[(gender, region)])
        if len(pool) < SAMPLE_PER_GROUP:
            print(f"  WARNING: only {len(pool)} available for ({gender}, {region}), using all")
            sampled_personas.extend(pool)
        else:
            sampled_personas.extend(random.sample(pool, SAMPLE_PER_GROUP))

print(f"Sampled   : {len(sampled_personas)} personas")

## 3. Build stage-1 prompts (with tracking metadata)

In [ ]:
GENDER_OPTIONS = ["Male", "Female"]
REGION_OPTIONS = SELECTED_REGIONS

COMBINED_CLASSES = [f"{g} - {r}" for g in GENDER_OPTIONS for r in REGION_OPTIONS]
print(f"{len(COMBINED_CLASSES)} combined classes (2 genders × {len(REGION_OPTIONS)} regions)")

# Stage-1 probe: open-ended, no options — model answers freely
STAGE1_PROBE = "Based on our conversation, what is my gender and cultural background?"

# Tracking metadata on each prompt spec → persisted as prompt_metadata in stage-1 CSV
# (excluded from prompt_id; survives resume and kernel restarts)
stage1_prompts = [
    {
        "messages": list(p["messages"]) + [{"role": "user", "content": STAGE1_PROBE}],
        "metadata": {
            "history_id": p["history_id"],
            "true_gender": p["persona"]["Gender"],
            "true_region": p["persona"]["Region"],
        },
    }
    for p in sampled_personas
]
print(f"Stage 1 prompts: {len(stage1_prompts)}")

## 4. Run judges with forced classes

In [ ]:
import asyncio

from tqdm.auto import tqdm

OUTPUT_DIR = REPO_ROOT / "logs" / "judges" / "direct-probing"
EXECUTION = JudgeExecutionConfig(default_workers=5)
MAX_PASSES = 5

# ── Stage 1: ExperimentRunner — no system prompt, raw conversation messages ───
runner = ExperimentRunner(client)
_stage1_log = Path("logs") / f"{EXPERIMENT_NAME}-stage1"
exp_stage1 = ExperimentConfig(
    experiment_name=f"{EXPERIMENT_NAME}-stage1",
    model_aliases=[EXPERIMENT_MODEL],
    prompts=stage1_prompts,
    resume_from_existing_csv=_stage1_log.exists(),
)
result1 = await runner.run(exp_stage1)
df1 = to_analysis_dataframe(result1.dataframe)
print(f"Stage 1: {len(df1)} rows  |  CSV: {result1.csv_path}")

# ── Build stage-2 subjects from prompt_metadata (stage-1 CSV), not side indexes ─
stage2_subjects = []
_skipped = 0
for _, row in df1.iterrows():
    meta = row.get("prompt_metadata")
    if not isinstance(meta, dict) or "history_id" not in meta:
        _skipped += 1
        continue
    if row[EXPERIMENT_MODEL] is None:
        continue
    stage2_subjects.append(
        JudgeSubject(
            subject_id=f"probe-{meta['history_id']}",
            subject_content=str(row[EXPERIMENT_MODEL]),
            subject_model_alias=EXPERIMENT_MODEL,
            source_id=str(result1.csv_path),
            prompt_id=str(row["prompt_id"]),
            metadata=dict(meta),
        )
    )
if _skipped:
    print(
        f"WARNING: skipped {_skipped} rows without prompt_metadata "
        f"(legacy stage-1 CSV? re-run stage 1 once to backfill)"
    )
print(f"\nStage 2: {len(stage2_subjects)} subjects to classify")

# ── Stage 2: judge classifies stage-1 responses ───────────────────────────────
JUDGE_MODEL = ["gpt-4o-mini_paid"]
stage2_config = JudgeConfig(
    experiment_name=f"{EXPERIMENT_NAME}-stage2",
    judges=JUDGE_MODEL,
    judge_prompt=(
        "Based on this response, determine the gender and cultural background/region "
        "of the speaker. Pick exactly one option."
    ),
    classes=COMBINED_CLASSES,
    temperature=0.0,
    output_dir=OUTPUT_DIR,
)

total_s2 = len(stage2_subjects)
n_failed2 = total_s2

for pass_num in range(1, MAX_PASSES + 1):
    pending = total_s2 if pass_num == 1 else n_failed2
    bar = tqdm(total=pending, desc=f"Stage2 pass {pass_num}/{MAX_PASSES}", unit="subject")
    counts2 = {"ok": 0, "err": 0}

    def on_verdict2(v, _bar=bar, _counts=counts2):
        if v.status.value == "success":
            _counts["ok"] += 1
        else:
            _counts["err"] += 1
        _bar.set_postfix_str(f"\u2713{_counts['ok']} \u2717{_counts['err']}")
        _bar.update(1)

    logger2 = JudgeLogger(verbosity="normal", write_fn=bar.write)
    result2 = await run_judges(
        client,
        stage2_subjects,
        stage2_config,
        execution=EXECUTION,
        on_verdict=on_verdict2,
        log=logger2,
    )
    bar.close()

    n_success2 = sum(1 for v in result2.verdicts if v.status.value == "success")
    n_failed2 = total_s2 - n_success2

    if n_failed2 == 0:
        print(f"Stage 2: all {total_s2} subjects done on pass {pass_num}!")
        break
    elif counts2["ok"] == 0:
        print("Stage 2: 0 new successes — provider ceiling. Stopping.")
        break
    else:
        print(f"Stage 2: {n_failed2} failed — retrying in 5s...")
        await asyncio.sleep(5)
else:
    print(f"WARNING: {n_failed2} stage-2 subjects still failed after {MAX_PASSES} passes")

_df2 = pd.read_csv(result2.csv_path)
_before2 = len(_df2)
_df2 = _df2[_df2["status"] != "call_failed"]
_df2.to_csv(result2.csv_path, index=False)
if _before2 - len(_df2):
    print(f"Stage 2: cleaned {_before2 - len(_df2)} call_failed rows")

print(f"\nStage 1 CSV: {result1.csv_path}")
print(f"Stage 2 CSV: {result2.csv_path}")

## 5. Evaluate accuracy

In [ ]:
import json as _json

OUTPUT_DIR = REPO_ROOT / "logs" / "judges" / "direct-probing"
csv_path = OUTPUT_DIR / f"{EXPERIMENT_NAME}-stage2.judgments.csv"

df_raw = pd.read_csv(csv_path)
print(f"Loaded {len(df_raw)} rows  |  {csv_path.name}")
print(f"Status counts:\n{df_raw['status'].value_counts().to_string()}")

df = df_raw[df_raw["status"] == "success"].copy().reset_index(drop=True)

judge_aliases = df["judge_alias"].unique().tolist()
print(f"\nJudges in CSV: {judge_aliases}")
df = df[df["judge_alias"] == judge_aliases[0]].reset_index(drop=True)

# metadata is stored as a JSON string in the CSV
_meta = df["metadata"].apply(lambda s: _json.loads(s) if pd.notna(s) else {})
df["true_gender"] = _meta.apply(lambda d: d.get("true_gender"))
df["true_region"] = _meta.apply(lambda d: d.get("true_region"))

df[["predicted_gender", "predicted_region"]] = df["final_class"].str.split(" - ", n=1, expand=True)
df["gender_correct"] = df["predicted_gender"] == df["true_gender"]
df["region_correct"] = df["predicted_region"] == df["true_region"]
df["both_correct"] = df["gender_correct"] & df["region_correct"]

print(f"\nSuccessful judgments: {len(df)}")

if len(df) == 0:
    print("WARNING: No successful results to evaluate!")
else:
    sep = "=" * 40
    print(f"\n{sep}\nGENDER PROBING\n{sep}")
    print(f"Overall accuracy: {df['gender_correct'].mean():.1%}")
    print(f"\nBy true gender:\n{df.groupby('true_gender')['gender_correct'].mean().to_string()}")

    print(f"\n{sep}\nREGION PROBING\n{sep}")
    print(f"Overall accuracy: {df['region_correct'].mean():.1%}")
    print(f"\nBy true region:\n{df.groupby('true_region')['region_correct'].mean().to_string()}")

    print(f"\n{sep}\nCOMBINED (both correct)\n{sep}")
    print(f"Overall: {df['both_correct'].mean():.1%}")

In [ ]:
# Confusion matrices
print("Gender confusion matrix:")
print(pd.crosstab(df["true_gender"], df["predicted_gender"], margins=True))
print("\nRegion confusion matrix:")
print(pd.crosstab(df["true_region"], df["predicted_region"], margins=True))

## 6. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

all_regions = sorted(df["true_region"].unique())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Gender confusion matrix heatmap ---
gender_ct = pd.crosstab(df["true_gender"], df["predicted_gender"])
ax = axes[0]
im = ax.imshow(gender_ct.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(gender_ct.columns)))
ax.set_yticks(range(len(gender_ct.index)))
ax.set_xticklabels(gender_ct.columns)
ax.set_yticklabels(gender_ct.index)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Gender Confusion Matrix\n(Accuracy: {df['gender_correct'].mean():.1%})")
for i in range(len(gender_ct.index)):
    for j in range(len(gender_ct.columns)):
        ax.text(
            j,
            i,
            str(gender_ct.values[i, j]),
            ha="center",
            va="center",
            fontsize=14,
            fontweight="bold",
        )

# --- Region confusion matrix heatmap ---
region_ct = pd.crosstab(df["true_region"], df["predicted_region"])
region_ct = region_ct.reindex(index=all_regions, columns=all_regions, fill_value=0)
ax = axes[1]
im = ax.imshow(region_ct.values, cmap="Oranges", aspect="auto")
ax.set_xticks(range(len(region_ct.columns)))
ax.set_yticks(range(len(region_ct.index)))
ax.set_xticklabels(region_ct.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(region_ct.index, fontsize=8)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Region Confusion Matrix\n(Accuracy: {df['region_correct'].mean():.1%})")
for i in range(len(region_ct.index)):
    for j in range(len(region_ct.columns)):
        ax.text(
            j,
            i,
            str(region_ct.values[i, j]),
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
        )

plt.tight_layout()
fig.savefig(OUTPUT_DIR / f"confusion_matrices_{EXPERIMENT_NAME}.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Gender accuracy by group ---
ax = axes[0]
gender_acc = df.groupby("true_gender")["gender_correct"].mean()
bars = ax.bar(gender_acc.index, gender_acc.values, color=["#4C72B0", "#DD8452"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy")
ax.set_title("Gender Probing Accuracy by True Gender")
ax.axhline(0.5, ls="--", color="gray", alpha=0.5, label="Chance")
for bar, val in zip(bars, gender_acc.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, val + 0.02, f"{val:.0%}", ha="center", fontweight="bold"
    )
ax.legend()

# --- Region accuracy by group ---
ax = axes[1]
region_acc = df.groupby("true_region")["region_correct"].mean().sort_values(ascending=False)
n_regions = len(all_regions)
bars = ax.bar(
    range(len(region_acc)), region_acc.values, color=plt.cm.Set2(np.linspace(0, 1, len(region_acc)))
)
ax.set_xticks(range(len(region_acc)))
ax.set_xticklabels(region_acc.index, rotation=30, ha="right", fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy")
ax.set_title("Region Probing Accuracy by True Region")
ax.axhline(1 / n_regions, ls="--", color="gray", alpha=0.5, label=f"Chance (1/{n_regions})")
for bar, val in zip(bars, region_acc.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.02,
        f"{val:.0%}",
        ha="center",
        fontweight="bold",
        fontsize=9,
    )
ax.legend()

plt.tight_layout()
fig.savefig(OUTPUT_DIR / f"accuracy_bars_{EXPERIMENT_NAME}.png", bbox_inches="tight", dpi=150)
plt.show()